In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from hoda.classification import ZScore, BTTDACV, SelectFdrMin1
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl

cv=StratifiedKFold(random_state=42, shuffle=True)
pipelines=dict()



clf = make_pipeline(
    FunctionTransformer(tl.to_numpy),
    StandardScaler(),
    SelectFdrMin1(alpha=0.05),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

bttda_params=dict(
        hoda_params=dict(
            max_iter=128,
            toeplitz=(1,),
            taper=False,
            verbose=False,
            refit_shrinkage=True
        ),
        clf=clf, verbose=False,
        cv=cv,
        n_jobs=10*5
)

pipelines['HODA'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=1,
        thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        **bttda_params)
    ),
    ('clf', clf)
])

pipelines['PARAFACDA'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=16,
        thetas=[0],
        **bttda_params)
    ),
    ('clf', clf)
])

pipelines['BTTDA'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=16,
        thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        **bttda_params)
    ),
    ('clf', clf)
])


In [3]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
    BNCI2014_008()
]

paradigm = P300(
    resample=48,

)
cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

In [4]:
import copy

job_args = []

for dataset in datasets:
    for subject in dataset.subject_list:
        for pipeline in pipelines:
            subj_dataset = copy.deepcopy(dataset)
            subj_dataset.subject_list = [subject]
            evaluation = WithinSessionEvaluation(
                paradigm=paradigm,
                datasets=subj_dataset,
                overwrite=True,
                random_state=42,
                n_jobs=5,
                suffix=f'bttda_dask',
                cache_config=cache_config
            )
            job_args.append((evaluation, pipeline))

def run_evaluation(evaluation, pipelien):
    return evaluation.process([pipeline])

In [5]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import pickle
from distributed.protocol.serialize import register_serialization_family


timeout = 6*60*60

def create_cluster():

    return SLURMCluster(
        cores=72,
        memory="100GB",
        account='llonpp',
        queue='batch',
        walltime='00:30:00',
        scheduler_options=dict(
            dashboard_address=':8787'
        ),
        job_extra_directives=[
            '-M wice',
        ],

        death_timeout=timeout,
        interface='ib0',
    )



def create_client(cluster):
    return Client(cluster)

In [6]:
#from dask.distributed import LocalCluster
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd

n_jobs = len(datasets)*max([len(d.subject_list) for d in datasets])*len(pipelines)
print(f'n_jobs: {n_jobs}')
with create_cluster() as cluster, create_client(cluster) as client:
    #cluster.adapt(minimum_jobs=1,maximum_jobs=100)
    cluster.scale(jobs=10)
    display.display(client)
    with joblib.parallel_backend('dask', wait_for_workers_timeout=timeout): 
        parallel = Parallel(n_jobs=n_jobs, verbose=10)
        results = parallel(delayed(run_evaluation)(*args) for args in job_args)
        results = pd.concat(results)



n_jobs: 24


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://172.23.6.138:8787/status,
Dashboard: http://172.23.6.138:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.23.6.138:38635,Workers: 0
Dashboard: http://172.23.6.138:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B



KeyboardInterrupt



Error in callback <bound method AutoreloadMagics.post_execute_hook of <IPython.extensions.autoreload.AutoreloadMagics object at 0x152e6278bb50>> (for post_execute), with arguments args (),kwargs {}:



KeyboardInterrupt



In [ ]:
results

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean')